In [2]:
%matplotlib widget

import xbox_proc # Structure, DateRange
import matplotlib.pyplot as plt

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from nptdms import TdmsFile
from datetime import datetime

### Define structure

In [3]:
#keys = ["Timestamp", "Line 1 Pressure", "Line 2 Pressure", "DUT 1 Room Temp", "DUT 2 Room Temp", "Pulse Count 2", "Load 2 Pressure", "DUT 2 Pressure", "StructureBSurf_In Temp", "StructureBSurf_Out Temp", "StructureBWater_In Temp", "Load2 Temp", "DUT 2 Flow", "Pulse Count 1", "Load 1 Pressure", "DUT 1 Pressure", "DUT 1 Surf In Temp", "DUT 1 Surf Out Temp", "DUT 1 Water In Temp", "Load 1 RF In Temp", "DUT 1 Flow"]
keys = ["Timestamp", 'Klystron A Pressure', 'Hybrid 1 Pressure', 'Pulse Compressor 1 Pressure', 'Line 1 Pressure', "DUT 1 Pressure",'Load 1 Pressure',
        '1PKIA_ampcursor Blue mean ', '1PKIB_ampcursor Blue mean ', '1PLIA_ampcursor Blue mean ', '1PLIB_ampcursor Blue mean ', '1PSI_ampcursor grey mean ', 
        'Gauge Klystron A Pressure', 'Gauge PC 1 Pressure', 'Gauge Line 1 Pressure',  'Gauge Load 1 Pressure', 'Gauge Klystron B Pressure', 'Gauge DUT 1 Pressure',
        'Repetition rate']

s = xbox_proc.Structure(
    name="Xbox3_TD26CIEMAT_L1",
    data_dir="//cernbox-drive/winspaces/x/xboxes/Xbox3_TD26CIEMAT_L1",
    xbox=3,
    stand=1,
   
    P_ref = 42.3,  # [MW]
    G_ref = 100,  # MV/m
    t_fill=57.25e-9,
    
    BDR_ref=1.7e-6,
    PL_ref=180, # ns

    trend_keys=keys
    )

### Define data range

In [ ]:
dr = xbox_proc.DateRange(20260101, 20260120)

### Extract data from tdms file to h5

In [4]:
event_df = s.extract_events(
    dr,
    out_h5="../data/event_data.h5",
    key="event_data",
    mode="w"   # overwrite for this test (append 'a' otherwise)
)

event_df.head()

,pulse_count,log_type,timestamp,PKIA_amp_total_power,PKIA_amp_length,PKIA_amp_peak,PKIA_amp_mean,PKIA_amp_start,PKIB_amp_total_power,PKIB_amp_length,...,PEIA_peak_length,PEIA_mean_flat,PEIA_start,PEIA_t,DC_up_total_A,DC_up_peak_A,DC_down_total_A,DC_down_peak_A,BD_struct,BD_loc
0,377253067,3,2026-01-10 00:01:10.198522,0.371312,8.225000e-07,461097.5742,248379.732766,7.175000e-07,0.415909,0.000001,...,1.812500e-08,1.128797e+06,6.612500e-07,0.0,-0.000306,-51,-0.000233,-34,0,0
1,377257060,3,2026-01-10 00:01:50.182441,0.013635,1.570625e-06,4754.2276,3754.912608,7.000000e-07,0.025224,0.000005,...,2.756250e-07,2.160770e+04,1.625000e-08,0.0,-0.000315,-52,-0.000266,-41,0,0
2,377261878,3,2026-01-10 00:02:30.198573,0.136521,8.268750e-07,163213.3612,87922.251903,7.175000e-07,0.133431,0.000001,...,1.312500e-08,3.900065e+05,6.631250e-07,0.0,-0.000315,-51,-0.000270,-44,0,0
3,377267887,3,2026-01-10 00:03:10.209398,0.255227,8.575000e-07,311588.5770,168680.349506,7.175000e-07,0.272018,0.000001,...,1.562500e-08,7.279421e+05,6.600000e-07,0.0,-0.000316,-51,-0.000272,-43,0,0
4,377273880,3,2026-01-10 00:03:50.192464,0.328195,8.575000e-07,405963.8326,218691.714363,7.131250e-07,0.359896,0.000001,...,2.062500e-08,9.740261e+05,6.612500e-07,0.0,-0.000313,-51,-0.000262,-41,0,0


### Load data from h5

In [6]:
# Load the event_data df in s object
s.load(path="../data/event_data.h5", key="event_data")
s.post_process()

df_event = s.df

### Example plotting

In [7]:
print(df_event.head())

   pulse_count  log_type                  timestamp  PKIA_amp_total_power  \
0    377253067         3 2026-01-10 00:01:10.198522              0.371312   
1    377257060         3 2026-01-10 00:01:50.182441              0.013635   
2    377261878         3 2026-01-10 00:02:30.198573              0.136521   
3    377267887         3 2026-01-10 00:03:10.209398              0.255227   
4    377273880         3 2026-01-10 00:03:50.192464              0.328195   

   PKIA_amp_length  PKIA_amp_peak  PKIA_amp_mean  PKIA_amp_start  \
0     8.225000e-07    461097.5742  248379.732766    7.175000e-07   
1     1.570625e-06      4754.2276    3754.912608    7.000000e-07   
2     8.268750e-07    163213.3612   87922.251903    7.175000e-07   
3     8.575000e-07    311588.5770  168680.349506    7.175000e-07   
4     8.575000e-07    405963.8326  218691.714363    7.131250e-07   

   PKIB_amp_total_power  PKIB_amp_length  ...  lost_power_A  cum_BD_Load_A  \
0              0.415909         0.000001  ...     

In [ ]:
plt.rcParams.update({
    'font.size': 8,
    'axes.titlesize': 10,
    'legend.fontsize': 8,
    'lines.linewidth': 1})

df_e_f = df_event.query("log_type!=2")
df_e_f = df_e_f.query("PKIA_amp_length>10e-9")

x = df_e_f["pulse_count"]/1e6 # M pulses

fig, ax = plt.subplots(3, 1, figsize=(8, 5), sharex=True, dpi=150)

ax[0].plot(x, df_e_f["PSIA_peak"], color = 'tab:blue')
ax[0].plot(x, df_e_f["PKIA_amp_mean"]*1e6, color = 'tab:red')

axx = ax[0].twinx()
axx.plot(x, df_e_f["cum_BD_DUT_withDC_A"], color = 'tab:orange')
axx.plot(x, df_e_f["cum_BD_HyPC_A"]-700, color = 'tab:green') #because I messed up
axx.plot(x, df_e_f["cum_BD_Load_A"], color = 'tab:gray')


ax[0].set_ylabel("PSIA [MW]")
axx.set_ylabel("Cum. BDs")

ax[0].set_ylim([0,5])
axx.set_ylim([0,2000])

axxx = ax[0].twinx()
axxx.plot(x, df_e_f["PKIA_amp_length"], color = 'black')
axxx.set_ylabel("PKI length [ns]")

ax[1].plot(x, df_e_f["BDR_DUT_A"], color='orange', marker=',', label = "BDR DUT A")
ax[1].plot(x, df_e_f["BDR_HyPC_A"], color='purple', marker=',', label = "BDR HyPC")
ax[1].plot(x, df_e_f["BDR_Load_A"], 'g', label = "BDR Load")
ax[1].legend()

ax[1].set_yscale('log')
ax[1].set_ylim([1e-7, 5e-4])
ax[1].set_ylabel("BDR [ns]")


# Conclude

ax[2].pcolormesh(xi/1e6,yi,zi,cmap="Blues",norm=mcolors.PowerNorm(0.5),alpha=1,shading='auto')
ax[2].contour(xi/1e6,yi,zi,cmap='gray',alpha=0.2,linewidths=0.5)
ax[2].set_ylabel("BDpos [ns]")
ax[2].set_xlabel(r"# pulses [$\times 10^6$]")

NameError: name 'plt' is not defined